In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

data_menu = [
    # Kategori Kopi
    {"nama_menu": "Espresso Tubruk", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "rekomendasi_barista": True},
    {"nama_menu": "Kopi Jos Jogja", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30},
    {"nama_menu": "Americano Aceh Gayo", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": True},
    {"nama_menu": "Latte Toraja", "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 2},
    {"nama_menu": "Avocado Coffee", "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 1},
    
    # Kategori Makanan
    {"nama_menu": "Nasi Goreng Rempah", "harga": 28000, "kategori": "Makanan", "pedas_level": 2},
    {"nama_menu": "Mie Goreng Jawa", "harga": 24000, "kategori": "Makanan", "pedas_level": 1},
    {"nama_menu": "Ayam Geprek Sambal Korek", "harga": 26000, "kategori": "Makanan", "pedas_level": 3, "rekomendasi_barista": True},
    {"nama_menu": "Roti Bakar Kaya", "harga": 18000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Pisang Goreng Keju", "harga": 15000, "kategori": "Makanan", "pedas_level": 0},
    {"nama_menu": "Singkong Goreng Merekah", "harga": 14000, "kategori": "Makanan", "pedas_level": 0}
]

koleksi_menu.insert_many(data_menu)
print("Berhasil menyimpan", koleksi_menu.count_documents({}), "menu ke database!")

Berhasil menyimpan 12 menu ke database!


In [3]:
# 1. Kopi dengan tingkat_kepahitan > 3
print("1. Menu Kopi dengan tingkat kepahitan > 3:")
for item in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"   - {item['nama_menu']} (Kepahitan: {item['tingkat_kepahitan']})")

# 2. Harga di antara 15.000 dan 30.000
print("\n2. Menu dengan harga antara Rp15.000 - Rp30.000:")
for item in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"   - {item['nama_menu']} (Harga: Rp{item['harga']:,})")

# 3. Tidak memiliki field pedas_level sama sekali
print("\n3. Menu yang TIDAK punya field 'pedas_level':")
for item in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"   - {item['nama_menu']} ({item['kategori']})")

1. Menu Kopi dengan tingkat kepahitan > 3:
   - Espresso Tubruk (Kepahitan: 5)
   - Kopi Jos Jogja (Kepahitan: 4)
   - Americano Aceh Gayo (Kepahitan: 4)

2. Menu dengan harga antara Rp15.000 - Rp30.000:
   - Espresso Tubruk (Harga: Rp18,000)
   - Kopi Jos Jogja (Harga: Rp20,000)
   - Americano Aceh Gayo (Harga: Rp22,000)
   - Kopi Susu Gula Aren (Harga: Rp25,000)
   - Latte Toraja (Harga: Rp28,000)
   - Nasi Goreng Rempah (Harga: Rp28,000)
   - Mie Goreng Jawa (Harga: Rp24,000)
   - Ayam Geprek Sambal Korek (Harga: Rp26,000)
   - Roti Bakar Kaya (Harga: Rp18,000)
   - Pisang Goreng Keju (Harga: Rp15,000)

3. Menu yang TIDAK punya field 'pedas_level':
   - Espresso Tubruk (Kopi)
   - Kopi Jos Jogja (Kopi)
   - Americano Aceh Gayo (Kopi)
   - Kopi Susu Gula Aren (Kopi)
   - Latte Toraja (Kopi)
   - Avocado Coffee (Kopi)


In [4]:
# Menambah field status: "Tersedia" ke SELURUH dokumen
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Ubah status 2 menu pilihan jadi "Habis"
koleksi_menu.update_one({"nama_menu": "Kopi Jos Jogja"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek Sambal Korek"}, {"$set": {"status": "Habis"}})

# Cek hasil update
print("Daftar status menu setelah diupdate:")
for item in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"   - {item['nama_menu']}: [{item['status']}]")

Daftar status menu setelah diupdate:
   - Espresso Tubruk: [Tersedia]
   - Kopi Jos Jogja: [Habis]
   - Americano Aceh Gayo: [Tersedia]
   - Kopi Susu Gula Aren: [Tersedia]
   - Latte Toraja: [Tersedia]
   - Avocado Coffee: [Tersedia]
   - Nasi Goreng Rempah: [Tersedia]
   - Mie Goreng Jawa: [Tersedia]
   - Ayam Geprek Sambal Korek: [Habis]
   - Roti Bakar Kaya: [Tersedia]
   - Pisang Goreng Keju: [Tersedia]
   - Singkong Goreng Merekah: [Tersedia]


In [6]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, round as spark_round

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Ambil data dari MongoDB ke pandas DataFrame
data_mongo = list(koleksi_menu.find({}))
df_pd = pd.DataFrame(data_mongo)

# 2. Pembersihan & Penyelarasan Tipe Data
# a. Ubah ObjectId MongoDB menjadi string
df_pd["_id"] = df_pd["_id"].astype(str)

# b. Tangani kolom boolean (misal: rekomendasi_barista) agar tidak bentrok dengan NaN/Double
if "rekomendasi_barista" in df_pd.columns:
    df_pd["rekomendasi_barista"] = df_pd["rekomendasi_barista"].fillna(False).astype(bool)

# c. Ubah NaN pada kolom numerik yang opsional menjadi None (null khas Python)
df_pd = df_pd.replace({np.nan: None})

# 3. Buat Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pd)

# Tampilkan Schema & Data
print("--- Spark Schema ---")
df_spark_menu.printSchema()

print("\n--- Tampilan Data Spark ---")
df_spark_menu.show()

# Analisis groupBy: Rata-rata harga per kategori
print("\n--- Rata-rata Harga per Kategori ---")
df_spark_menu.groupBy("kategori").agg(
    spark_round(avg("harga"), 2).alias("rata_rata_harga")
).show()

spark.stop()

/tmp/ipykernel_12098/403021445.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pd["rekomendasi_barista"] = df_pd["rekomendasi_barista"].fillna(False).astype(bool)


--- Spark Schema ---
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: double (nullable = true)
 |-- pedas_level: double (nullable = true)


--- Tampilan Data Spark ---


+--------------------+--------------------+-----+--------+-----------------+-------------------+--------+----+-----------+
|                 _id|           nama_menu|harga|kategori|tingkat_kepahitan|rekomendasi_barista|  status|stok|pedas_level|
+--------------------+--------------------+-----+--------+-----------------+-------------------+--------+----+-----------+
|6ae31a20ba1231158...|     Espresso Tubruk|18000|    Kopi|              5.0|               true|Tersedia|NULL|       NULL|
|6ae31a20ba1231158...|      Kopi Jos Jogja|20000|    Kopi|              4.0|              false|   Habis|30.0|       NULL|
|6ae31a20ba1231158...| Americano Aceh Gayo|22000|    Kopi|              4.0|              false|Tersedia|NULL|       NULL|
|6ae31a20ba1231158...| Kopi Susu Gula Aren|25000|    Kopi|              2.0|               true|Tersedia|NULL|       NULL|
|6ae31a20ba1231158...|        Latte Toraja|28000|    Kopi|              2.0|              false|Tersedia|NULL|       NULL|
|6ae31a20ba12311

Mengonversi data dari MongoDB ke Spark DataFrame memiliki tantangan utama pada ketidakseragaman skema (flexible schema), di mana setiap dokumen memiliki field yang berbeda-beda. Hal ini memicu konflik tipe data saat dikonversi melalui Pandas — seperti bentrok antara BooleanType dan DoubleType akibat nilai NaN.   Solusinya adalah melakukan pembersihan data awal (data preprocessing) pada Pandas DataFrame sebelum dikonversi ke Spark DataFrame. Langkah yang dilakukan meliputi mengubah ObjectId menjadi string, mengisi nilai NaN pada kolom boolean dengan False, serta mengganti np.nan menjadi None agar PySpark dapat menetapkan tipe data secara konsisten dan membacanya sebagai nilai null.